In [1]:
from file_aggregation import file_aggregation
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
X, y, groups = file_aggregation('/Users/reymendoza/Downloads/mit-bih-arrhythmia-database-1.0.0')

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

Found 46 valid records.
Excluded records: ['102', '104']
Creating RawArray with float64 data, n_channels=1, n_times=650160
    Range : 0 ... 650159 =      0.000 ...  1805.997 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=650160
    Range : 0 ... 650159 =      0.000 ...  1805.997 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=650160
    Range : 0 ... 650159 =      0.000 ...  1805.997 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=650160
    Range : 0 ... 650159 =      0.000 ...  1805.997 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=650160
    Range : 0 ... 650159 =      0.000 ...  1805.997 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=650160
    Range : 0 ... 650159 =      0.000 ...  1805.997 secs
Ready.
Creating RawArray with float64 data, n_channels=1, n_times=650160
    Range : 0 ... 650159 =      0.000 ...  1805.997 secs
Ready.
Creating RawArray with float64 da

In [2]:
from sklearn.model_selection import StratifiedGroupKFold
from cnn_model import ECGCNN1D
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

gkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
split = gkf.split(X.numpy(), y.squeeze(1).numpy(), groups=groups)

fold_results = []

for fold, (train_idx, test_idx) in enumerate(split, start=1):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    print(f"Fold {fold}")

    train_ds = TensorDataset(X_train, y_train)
    test_ds = TensorDataset(X_test, y_test)

    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

    model = ECGCNN1D()

    n_pos = y_train.sum().item()
    n_neg = len(y_train) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)

    epochs = 5

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for xb, yb in train_loader:

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * xb.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)

        model.eval()

        all_probs = []
        all_preds = []
        all_true = []

        with torch.no_grad():
            for xb, yb in test_loader:

                logits = model(xb)
                probs = torch.sigmoid(logits)
                preds = (probs >= 0.5).float()

                all_probs.append(probs)
                all_preds.append(preds)
                all_true.append(yb)

        y_prob = torch.cat(all_probs).numpy().ravel()
        y_pred = torch.cat(all_preds).numpy().ravel()
        y_true = torch.cat(all_true).numpy().ravel()

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        auc = roc_auc_score(y_true, y_prob)
        cm = confusion_matrix(y_true, y_pred)

        print(
        f"Epoch {epoch+1:02d} | "
        f"Loss: {epoch_loss:.4f} | "
        f"Acc: {acc:.4f} | "
        f"Prec: {prec:.4f} | "
        f"Rec: {rec:.4f} | "
        f"F1: {f1:.4f} | "
        f"AUC: {auc:.4f}"
    )

    fold_results.append({
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc
    })

print(fold_results)

Fold 1
Epoch 01 | Loss: 0.8776 | Acc: 0.6617 | Prec: 0.2755 | Rec: 0.1635 | F1: 0.2052 | AUC: 0.4865
Epoch 02 | Loss: 0.8626 | Acc: 0.6617 | Prec: 0.2755 | Rec: 0.1635 | F1: 0.2052 | AUC: 0.4012
Epoch 03 | Loss: 0.8592 | Acc: 0.6617 | Prec: 0.2755 | Rec: 0.1635 | F1: 0.2052 | AUC: 0.3961
Epoch 04 | Loss: 0.8573 | Acc: 0.6617 | Prec: 0.2755 | Rec: 0.1635 | F1: 0.2052 | AUC: 0.4942
Epoch 05 | Loss: 0.8534 | Acc: 0.6617 | Prec: 0.2755 | Rec: 0.1635 | F1: 0.2052 | AUC: 0.4217
Fold 2
Epoch 01 | Loss: 0.8956 | Acc: 0.4145 | Prec: 0.2874 | Rec: 0.8910 | F1: 0.4347 | AUC: 0.5976
Epoch 02 | Loss: 0.8822 | Acc: 0.4145 | Prec: 0.2874 | Rec: 0.8910 | F1: 0.4347 | AUC: 0.5974
Epoch 03 | Loss: 0.8729 | Acc: 0.4145 | Prec: 0.2874 | Rec: 0.8910 | F1: 0.4347 | AUC: 0.5162
Epoch 04 | Loss: 0.8703 | Acc: 0.4145 | Prec: 0.2874 | Rec: 0.8910 | F1: 0.4347 | AUC: 0.5976
Epoch 05 | Loss: 0.8677 | Acc: 0.4145 | Prec: 0.2874 | Rec: 0.8910 | F1: 0.4347 | AUC: 0.5229
Fold 3
Epoch 01 | Loss: 0.9386 | Acc: 0.4584 |

In [3]:
cm

array([[15007,  1687],
       [ 4401,   424]])

In [4]:
fold_results

[{'accuracy': 0.6616525860292602,
  'precision': 0.27550357374918777,
  'recall': 0.16351716158889318,
  'f1': 0.20522749273959343,
  'auc': 0.4217141254376963},
 {'accuracy': 0.4144681300126636,
  'precision': 0.28744693753790174,
  'recall': 0.8909774436090225,
  'f1': 0.4346629986244842,
  'auc': 0.5228650883303524},
 {'accuracy': 0.45842405391394503,
  'precision': 0.023784015427469468,
  'recall': 0.01849383538820393,
  'f1': 0.020807948261317835,
  'auc': 0.5652424257136042},
 {'accuracy': 0.6077646951857072,
  'precision': 0.38534959472767216,
  'recall': 0.9082306730940903,
  'f1': 0.5411126475906819,
  'auc': 0.6088868549289742},
 {'accuracy': 0.7170872252428087,
  'precision': 0.2008526764566556,
  'recall': 0.08787564766839379,
  'f1': 0.12226066897347174,
  'auc': 0.364558697828825}]